In [1]:
import os
# Replace with your actual proxy URL and port
proxy_url = "http://proxy-dku.oit.duke.edu:3128"
os.environ["HTTP_PROXY"] = proxy_url
os.environ["HTTPS_PROXY"] = proxy_url
import cocopp

In [4]:
from cocopp import archiving
local_path = 'bbob-exp'
archiving.create(local_path)
cocopp.testbedsettings.GECCOBBOBTestbed.settings['instancesOfInterest'] = [1,2,3,4,5]
cocopp.config.config()

In [5]:
my_archive = archiving.get(local_path)
data = cocopp.main(my_archive.get_all(''))

ValueError: The folder "/data1/home/jw1017/AS_BBO/AS_BBOB_SOO/data_generation/performances/bbob-exp" seems not to "be" a COCO data archive as it does not contain a coco_archive_definition.txt file).
Use `create(folder)` or `get(URL)` of `cocopp.archiving` to create/get this file.

In [ ]:
import numpy as np
# Iterate over the DictAlg object:
erts = []
algos = []
for alg_info, dataset_list in data.items():
    alg_name = alg_info[0]  # Algorithm name, e.g., 'BrentSTEPrr_Posik'
    if alg_name not in algos:
        algos.append(alg_name)
    print(f"Algorithm: {alg_name}")
    # Iterate over the DataSet objects in the dataset_list
    for ds in dataset_list:
        if alg_name=='BrentSTEPqi_Posik' and ds.dim in [2,3,5,10]:
            instances = ds.instancenumbers
            _, unique_indices = np.unique(instances, return_index=True)
            print(f'dim={ds.dim}, func={ds.funcId}')
            print(ds.funcId)
            print(ds.detEvals([1e-2])[0])
            print(ds._detMaxEvals(1e-2))

# Get IID performance

In [ ]:
import numpy as np
import pandas as pd

performance_records = []  # To store one dict per instance

# Dimensions to include (extend as needed)
dims = [2, 3, 5, 10, 20, 40]

for alg_info, dataset_list in data.items():
    alg_name = alg_info[0]  # e.g., 'BrentSTEPrr_Posik'
    
    for ds in dataset_list:
        if ds.dim in dims:
            # Get the evaluations and successes
            evals = ds._detMaxEvals(0.01)
            su_evals = ds.detEvals([0.01])[0]
            
            # Get instance numbers and unique indices to filter duplicate instances if any
            instances = ds.instancenumbers
            _, unique_indices = np.unique(instances, return_index=True)
            
            filtered_evals = evals[unique_indices]
            filtered_su_evals = su_evals[unique_indices]
            success = ~np.isnan(filtered_su_evals)
            
            # Special handling for BrentSTEPqi_Posik
            if alg_name == 'BrentSTEPqi_Posik':
                maxevals = pd.read_csv(
                    f"bbob-exp/BrentSTEPqi/data_f{ds.funcId}/bbobexp_f{ds.funcId}_DIM{ds.dim}.dat",
                    delim_whitespace=True,
                )
                max_evals = maxevals.iloc[:, 0].to_list()
                result = [max_evals[i - 1] for i in range(1, len(max_evals)) if max_evals[i] == '%' ][:5]
                for i in range(len(filtered_evals)):
                    if not success[i]:
                        filtered_evals[i] = float(result[i])
            
            # Save performance for each unique instance
            for idx, pos in enumerate(unique_indices):
                record = {
                    "algo": ds.algId,
                    "funcId": ds.funcId,
                    "Dim": ds.dim,
                    "iid": instances[pos],
                    "evals": filtered_evals[idx],
                    "success": success[idx],
                }
                performance_records.append(record)

# Create a DataFrame from the performance records
performance_df = pd.DataFrame(performance_records)
# Optionally, save the results to a CSV file:
performance_df.to_csv("performance_records.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd

# Assume performance_df has the columns: funcId, Dim, iid, evals, and success.
# If performance_df might contain duplicate rows for the same instance,
# we drop duplicates for the purpose of computing the group sum.
performance_df = pd.read_csv('performance_records.csv')
unique_instances = performance_df.drop_duplicates(subset=['algo','funcId', 'Dim', 'iid'])

# Compute group-level success sum for each (funcId, Dim) group,
# counting each unique instance (iid) only once.
group_success = unique_instances.groupby(['algo','funcId', 'Dim'])['success'].sum().reset_index()
group_success = group_success.rename(columns={'success': 'group_success_sum'})

# Merge the group success sum back into the original DataFrame.
merged_df = performance_df.merge(group_success, on=['algo','funcId', 'Dim'], how='left')

# Avoid division by zero: if group_success_sum is zero, set avg_evals to NaN.
merged_df.loc[merged_df['group_success_sum'] == 0, 'avg_evals'] = np.nan
merged_df.loc[merged_df['group_success_sum'] != 0, 'avg_evals'] = merged_df.loc[merged_df['group_success_sum'] != 0, 'evals'] / merged_df.loc[merged_df['group_success_sum'] != 0, 'group_success_sum']

# Keep only the required columns.
result_df = merged_df[['algo','funcId', 'Dim', 'iid', 'avg_evals']]

#print(result_df.head())

result = result_df.fillna(result_df.avg_evals.max())
df_wide = result.pivot(index=['funcId', 'Dim', 'iid'], columns='algo', values='avg_evals').reset_index()
df_wide.to_csv('iid_perform.csv', index=False)

# Get the original relERT benchmark

In [ ]:
erts = []
algos = []

# Dimensions to include (extend as needed)
dims = [2, 3, 5, 10, 20, 40]

for alg_info, dataset_list in data.items():
    alg_name = alg_info[0]  # Algorithm name, e.g., 'BrentSTEPrr_Posik'
    if alg_name not in algos:
        algos.append(alg_name)
    
    # Iterate over the DataSet objects in the dataset_list
    for ds in dataset_list:
        if ds.dim in dims:
            evals = ds._detMaxEvals(0.01)
            su_evals = ds.detEvals([0.01])[0]
            instances = ds.instancenumbers
            _, unique_indices = np.unique(instances, return_index=True)
            filtered_evals = evals[unique_indices]
            filtered_su_evals = su_evals[unique_indices]
            success = ~np.isnan(filtered_su_evals)
            if alg_name=='BrentSTEPqi_Posik':
                maxevals = pd.read_csv(f"bbob-exp/BrentSTEPqi/data_f{ds.funcId}/bbobexp_f{ds.funcId}_DIM{ds.dim}.dat", delim_whitespace=True)
                max_evals = maxevals.iloc[:, 0].to_list()
                result = [max_evals[i - 1] for i in range(1, len(max_evals)) if max_evals[i] == '%' ][:5]
                for i in range(len(filtered_evals)):
                    if success[i]==False:
                        filtered_evals[i]=float(result[i])
            if sum(success)==0:
                ert_value = np.nan
            else:
                ert_value = sum(filtered_evals)/sum(success)
            erts.append(ert_value)

import pandas as pd
func_ids = range(1, 25)

rows = []
index = 0

# The ordering is assumed to be: for each algo, for each dim, for each funcID
for algo in algos:
    for dim in dims:
        for func_id in func_ids:
            rows.append({'Algo': algo, 'Problem': f'f{func_id}', 'Dim': dim, 'ERT': erts[index]})
            index += 1

# Create a DataFrame from the rows
df = pd.DataFrame(rows)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df

In [ ]:
df_dim2 = df[df["Dim"] == 3]

# Pivot wider: rows are problems, columns are algo, values are ERT
df_wide = df_dim2.pivot(index="Problem", columns="Algo", values="ERT")

# Calculate the rel_ert for each row (ERT / ERT_best)
# ERT_best is the minimum ERT value in each row
df_wide["ERT_best"] = df_wide.min(axis=1)  # Best (minimum) ERT for each problem
df_wide.iloc[:,:-1] = df_wide.div(df_wide["ERT_best"], axis=0)  # Compute relative ERT (rel_ert)

# Fill missing values
df_wide = df_wide.fillna(36690.3)

# Round all numeric values to two decimals
#df_wide = df_wide.round(1)

# Drop the ERT_best column since it's not needed anymore
df_wide = df_wide.drop(columns=["ERT_best"])

# Create the grouping index based on the problem
problem_groups = {
    "f1-f5": ['f1', 'f2', 'f3', 'f4', 'f5'],
    "f6-f9": ['f6', 'f7', 'f8', 'f9'],
    "f10-f14": ['f10', 'f11', 'f12', 'f13', 'f14'],
    "f15-f19": ['f15', 'f16', 'f17', 'f18', 'f19'],
    "f20-f24": ['f20', 'f21', 'f22', 'f23', 'f24'],
    "all": ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24']
}

# Calculate the average of rel_ert for each group
rel_ert_avg = {}
for group_name, problems in problem_groups.items():
    group_df = df_wide.loc[problems]  # Select the rows corresponding to the current problem group
    rel_ert_avg[group_name] = group_df.mean(axis=0).round(1)  # Calculate the average of rel_ert for each algo in the group

# Convert the result to a DataFrame
result_df = pd.DataFrame(rel_ert_avg)
order = ['BrentSTEPqi_Posik', 'BrentSTEPrr_Posik', 'CMA-CSA_Atamna', 'fmincon_pal_noiseless', 'fminunc_pal_noiseless', 'HCMA_loshchilov_noiseless', 'HMLSL_pal_noiseless',
         'IPOP400D_auger_noiseless', 'MCS_huyer_noiseless', 'M_LSL_pal_noiseless', 'OQNLP_pal_noiseless', 'SMAC-BBOB_hutter_noiseless']
result_table = result_df.T[order]
result_table

In [ ]:
df["relERT"] = df["ERT"] / df.groupby(["Problem", "Dim"])["ERT"].transform("min")
df = df.fillna(36690.3)

In [ ]:
order = [f'f{i}' for i in range(1, 25)]
df_wide = df.pivot(index=["Problem", 'Dim'], columns="Algo", values="relERT").loc[order]
df_wide

In [ ]:
# benchmark
df_wide.to_csv('original.csv')

In [ ]:
df_dim2 = df[df["Dim"] == 3]

# Pivot wider: rows are problems, columns are algo, values are ERT
df_wide = df_dim2.pivot(index="Problem", columns="Algo", values="relERT")
# Fill missing values
df_wide = df_wide.fillna(36690.3)

# Create the grouping index based on the problem
problem_groups = {
    "f1-f5": ['f1', 'f2', 'f3', 'f4', 'f5'],
    "f6-f9": ['f6', 'f7', 'f8', 'f9'],
    "f10-f14": ['f10', 'f11', 'f12', 'f13', 'f14'],
    "f15-f19": ['f15', 'f16', 'f17', 'f18', 'f19'],
    "f20-f24": ['f20', 'f21', 'f22', 'f23', 'f24'],
    "all": ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24']
}

# Calculate the average of rel_ert for each group
rel_ert_avg = {}
for group_name, problems in problem_groups.items():
    group_df = df_wide.loc[problems]  # Select the rows corresponding to the current problem group
    rel_ert_avg[group_name] = group_df.mean(axis=0).round(1)  # Calculate the average of rel_ert for each algo in the group

# Convert the result to a DataFrame
result_df = pd.DataFrame(rel_ert_avg)
order = ['BrentSTEPqi_Posik', 'BrentSTEPrr_Posik', 'CMA-CSA_Atamna', 'fmincon_pal_noiseless', 'fminunc_pal_noiseless', 'HCMA_loshchilov_noiseless', 'HMLSL_pal_noiseless',
         'IPOP400D_auger_noiseless', 'MCS_huyer_noiseless', 'M_LSL_pal_noiseless', 'OQNLP_pal_noiseless', 'SMAC-BBOB_hutter_noiseless']
result_table = result_df.T[order]
result_table